# PhonePe Pulse Data Analysis & Machine Learning Pipeline
This notebook demonstrates a comprehensive data analysis and machine learning pipeline using the PhonePe Pulse transaction insights dataset.


## 1. Setup & Mock Data Generation
We import the necessary libraries and generate the single core dataset representing Aggregated Transactions for analysis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# Generate a single core dataset (Aggregated Transactions)
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'State': np.random.choice(['Maharashtra', 'Karnataka', 'Tamil Nadu', 'Delhi', 'UP'], n),
    'Year': np.random.choice([2021, 2022, 2023, 2024], n),
    'Quarter': np.random.choice([1, 2, 3, 4], n),
    'Type': np.random.choice(['P2P', 'Merchant', 'Recharge', 'Financial'], n),
    'Count': np.random.randint(1000, 500000, n),
    'Amount': np.random.uniform(1e5, 5e9, n)
})
df.head()


## 2. Data Cleaning & Wrangling
In this section, we drop duplicate records, impute missing values (using numeric median), cap outliers using the IQR method, and calculate the Average Transaction Value (ATV).


In [ ]:
# Handle missing values and drop duplicates
df = df.drop_duplicates().fillna(df.median(numeric_only=True))

# Cap outliers using the IQR method
Q1, Q3 = df['Amount'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df['Amount'] = df['Amount'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# Feature Engineering
df['Avg_Transaction_Value'] = df['Amount'] / df['Count']
df.head()


## 3. Exploratory Data Analysis (EDA)
We visualize the trend of transactions over time and the distribution of transaction volume across different states.


In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total Volume over Time
df.groupby(['Year', 'Quarter'])['Amount'].sum().plot(ax=axes[0], marker='o', title='Transaction Trend')
axes[0].set_ylabel('Amount (INR)')

# Plot 2: State-wise Volume
df.groupby('State')['Amount'].sum().sort_values().plot(kind='barh', ax=axes[1], title='Volume by State', color='teal')
plt.tight_layout()
plt.show()


## 4. ML Part A: Clustering (K-Means)
We cluster the states into 3 segments based on their average transaction metrics: `Amount`, `Count`, and `Avg_Transaction_Value`.


In [ ]:
# Cluster states based on average transaction metrics
state_metrics = df.groupby('State')[['Amount', 'Count', 'Avg_Transaction_Value']].mean()

# Fit K-Means
state_metrics['Cluster'] = KMeans(n_clusters=3, random_state=42).fit_predict(state_metrics)
print("--- State Clusters ---")
print(state_metrics[['Amount', 'Cluster']])


## 5. ML Part B: Prediction (Random Forest)
We encode categorical features, split the data into training and validation sets, and build a Random Forest Regressor to predict the transaction `Amount`.


In [ ]:
# One-hot encode categorical variables for regression
X = pd.get_dummies(df[['State', 'Type', 'Year', 'Quarter', 'Count']], drop_first=True)
y = df['Amount']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
preds = rf.predict(X_test)
print("\n--- Model Evaluation ---")
print(f"MAE:  {mean_absolute_error(y_test, preds):,.2f}")
print(f"R² Score: {r2_score(y_test, preds):.4f}")
